In [1]:
import threading
import queue
from concurrent.futures import ThreadPoolExecutor

class DataProcessor:
    def __init__(self, data_generator, num_workers, file_path):
        self.data_generator = data_generator
        self.num_workers = num_workers
        self.file_path = file_path
        self.output_queue = queue.Queue(maxsize=100)
        
    def _process_data(self, data):
        # process the data here
        processed_data = data * 2
        return processed_data
    
    def _write_to_file(self):
        with open(self.file_path, 'w') as f:
            while True:
                data = self.output_queue.get()
                
                # Check if there's any more data to be written
                if data is None:
                    break
                
                # Write the processed data to file
                f.write(str(data) + '\n')
                f.flush()
                
    def process_and_write_data(self):
        # Create a thread for writing the processed data to file
        write_thread = threading.Thread(target=self._write_to_file)
        write_thread.start()
        
        # Create a thread pool with the specified number of worker threads
        with ThreadPoolExecutor(max_workers=self.num_workers) as executor:
            # Generate and process data in parallel using the worker threads
            for data in self.data_generator:
                future = executor.submit(self._process_data, data)
                future.add_done_callback(lambda f: self.output_queue.put(f.result()))
            
            # Wait for all worker threads to finish before terminating the write thread
            executor.shutdown(wait=True)
            self.output_queue.put(None)
            write_thread.join()

In [2]:
# Define a generator function that generates random numbers between 1 and 10
def data_generator(number=10000):
    for i in range(number):
        yield i

# Create a DataProcessor instance with 4 worker threads and a file path of "output.txt"
processor = DataProcessor(data_generator(), num_workers=4, file_path="output.txt")

# Process and write the data to file
processor.process_and_write_data()

In [3]:
# importing cProfile
import cProfile
cProfile.run('processor.process_and_write_data()')

         70 function calls in 0.003 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.003    0.003 821007336.py:30(process_and_write_data)
        1    0.000    0.000    0.003    0.003 <string>:1(<module>)
        1    0.000    0.000    0.000    0.000 _base.py:632(__enter__)
        1    0.000    0.000    0.000    0.000 _base.py:635(__exit__)
        1    0.000    0.000    0.000    0.000 _weakrefset.py:38(_remove)
        1    0.000    0.000    0.000    0.000 _weakrefset.py:81(add)
        1    0.000    0.000    0.000    0.000 queue.py:121(put)
        1    0.000    0.000    0.000    0.000 queue.py:208(_qsize)
        1    0.000    0.000    0.000    0.000 queue.py:212(_put)
        1    0.000    0.000    0.000    0.000 thread.py:120(__init__)
        2    0.000    0.000    0.000    0.000 thread.py:230(shutdown)
        1    0.000    0.000    0.001    0.001 threading.py:1017(_wait_for_tstate_lo

In [4]:
import queue
import threading
from concurrent.futures import ThreadPoolExecutor

class DataProcessor:
    def __init__(self, generator, num_threads, output_file):
        self.generator = generator
        self.num_threads = num_threads
        self.output_file = output_file
        self.in_queue = queue.Queue()
        self.out_queue = queue.Queue()
        self.pool = ThreadPoolExecutor(max_workers=num_threads)
        self.write_thread = threading.Thread(target=self.write_to_file)
        self.write_thread.daemon = True

    def start(self):
        self.write_thread.start()
        while True:
            try:
                data = next(self.generator)
                self.in_queue.put(data)
                self.pool.submit(self.process_data, data)
            except StopIteration:
                break

    def process_data(self, data):
        # Implement data processing logic here
        result = data.upper()  # Example processing logic
        self.out_queue.put(result)

    def write_to_file(self):
        with open(self.output_file, 'w') as f:
            while True:
                try:
                    result = self.out_queue.get(timeout=1)
                    f.write(result + '\n')
                    self.out_queue.task_done()
                except queue.Empty:
                    if not self.pool._work_queue.empty():
                        continue
                    else:
                        break

    def stop(self):
        self.pool.shutdown(wait=True)
        self.out_queue.join()

In [5]:
processor = DataProcessor(data_generator(), num_threads=4, output_file='output.txt')

In [6]:
import cProfile
cProfile.run('processor.start()')

         251687 function calls in 0.873 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    10001    0.005    0.000    0.005    0.000 3386185446.py:2(data_generator)
        1    0.056    0.056    0.873    0.873 36788709.py:16(start)
        1    0.000    0.000    0.873    0.873 <string>:1(<module>)
    10000    0.016    0.000    0.047    0.000 _base.py:316(__init__)
        4    0.000    0.000    0.000    0.000 _weakrefset.py:81(add)
    10000    0.037    0.000    0.097    0.000 queue.py:121(put)
    10000    0.005    0.000    0.008    0.000 queue.py:212(_put)
    10000    0.054    0.000    0.709    0.000 thread.py:158(submit)
    10000    0.013    0.000    0.560    0.000 thread.py:193(_adjust_thread_count)
    10000    0.008    0.000    0.008    0.000 thread.py:46(__init__)
        4    0.000    0.000    0.000    0.000 threading.py:1095(daemon)
        4    0.000    0.000    0.000    0.000 threading.py:1110(daemon)
      

In [ ]:
processor.start()
processor.stop()